# Release notebook

This output-free copy corresponds to the final reported protocol. Exploratory results are not part of this repository. Use the frozen settings and artifact hashes in `configs/final_experiment.json`.


# 1단계: 최종 DistilBERT로 독립 9,000건 평가

GPU 런타임을 선택하고 위에서부터 실행합니다. Drive 연결 시 제한 재현 자료에 접근 권한이 있는 계정을 선택합니다.

- 최종 seed42 저장 모델로 **추론만** 합니다. 학습, 온도 재보정, 임계값 탐색은 하지 않습니다.
- T = 1.4801950079829693, 임계값 = 0.70, 최대 길이 = 256, 평가 배치 = 64, fp32.
- 최종 모델과 9,000건 데이터를 제한 Drive 재현 번들에서 가져오고 SHA256을 검증합니다. 접근 정보는 환경변수로만 제공합니다.
- 모델·토크나이저·데이터의 SHA256을 확인합니다. 예전 모델은 사용하지 않습니다.
- 새로운 출력 폴더를 사용합니다. 기존 12,000건, Mixed Emotion, 예전 9,000건 결과는 변경하지 않습니다.
- 다음 단계는 이번에 라우팅된 사례에 대한 Llama 3 Direct / CoT / 최종 v6c입니다. 이 노트북은 LLM을 실행하지 않습니다.

이 9,000건은 최종 Phase 1 설정과 Llama 3 프로토콜을 고정한 뒤 처음 평가한 별도 보류 세트입니다.


## 1. 패키지 설치
새 런타임에서도 이 셀부터 실행합니다. 이 분류 단계에는 HF_TOKEN과 bitsandbytes가 필요하지 않습니다.

In [ ]:
%pip install -q transformers==5.17.0 pandas==2.2.3 numpy==2.1.3 scipy==1.16.3 scikit-learn==1.6.1 tqdm safetensors


## 2. Drive 연결 및 파일 자동 준비
Drive 연결과 추가 인증에서 제한 재현 번들에 접근 권한이 있는 계정을 선택합니다. 처음 한 번만 약 250MB를 내려받고, 다음부터 저장된 파일을 검증해 재사용합니다. 파일 ID는 저장소에 기록하지 않고 `CGSLR_ASSET_PART1_FILE_ID` 등의 환경변수로 제공합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
OUTPUT_DIR = DRIVE_ROOT / "confidence_guided_llm_reasoning/outputs_final/unified_final/additional_9000_20260930/phase1"


In [ ]:
"""Download a private, hash-pinned model/data bundle in Colab."""

import hashlib
import os
import shutil
import zipfile
from pathlib import Path


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def verify_assets(root, expected_files):
    return all((root / name).is_file() and file_sha256(root / name) == digest
               for name, digest in expected_files.items())


def extract_verified_assets(archive_path, root, archive_sha256, expected_files):
    if file_sha256(archive_path) != archive_sha256:
        raise ValueError("Asset download SHA256 mismatch; no files extracted.")
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(expected_files):
            raise ValueError("Unexpected or duplicate files in the asset ZIP.")
        for name in names:
            relative = Path(name)
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError("Unsafe asset ZIP path.")
        for name in names:
            target = root / name
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name + ".partial")
            with archive.open(name) as source, temporary.open("wb") as output:
                shutil.copyfileobj(source, output, length=1024 * 1024)
            if file_sha256(temporary) != expected_files[name]:
                raise ValueError(f"Asset file hash mismatch: {name}")
            temporary.replace(target)
    if not verify_assets(root, expected_files):
        raise ValueError("Asset verification failed after extraction.")


def prepare_assets(drive_root, parts, archive_sha256, expected_files):
    root = drive_root / "confidence_guided_llm_reasoning/assets/final_seed42_existing9000_20260930"
    root.mkdir(parents=True, exist_ok=True)
    if verify_assets(root, expected_files):
        print("Verified saved final model and 9000-post data. No download needed.")
        return root / "data", root / "seed_42_model"
    archive_path = root / "final_seed42_existing9000_assets.zip"
    if not archive_path.is_file() or file_sha256(archive_path) != archive_sha256:
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession

        print("Authorize an account with access to the restricted reproduction bundle.")
        auth.authenticate_user()
        credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
        with AuthorizedSession(credentials) as session:
            for index, part in enumerate(parts, start=1):
                target = root / f"asset.part{index}"
                if target.is_file() and file_sha256(target) == part["sha256"]:
                    continue
                partial = target.with_suffix(target.suffix + ".partial")
                if not part["file_id"]:
                    raise RuntimeError(
                        f"Missing Drive file ID for asset part {index}. Set the documented "
                        f"CGSLR_ASSET_PART{index}_FILE_ID environment variable."
                    )
                url = f"https://www.googleapis.com/drive/v3/files/{part['file_id']}"
                with session.get(url, params={"alt": "media"}, stream=True, timeout=180) as response:
                    if response.status_code in (401, 403, 404):
                        raise RuntimeError(
                            "Restricted asset access failed. Authenticate with an authorized account; "
                            "do not change the model or download an old checkpoint."
                        )
                    response.raise_for_status()
                    from tqdm.auto import tqdm
                    with partial.open("wb") as handle, tqdm(total=part["size"], unit="B", unit_scale=True,
                                                            desc=f"Asset {index}/{len(parts)}") as progress:
                        for block in response.iter_content(chunk_size=1024 * 1024):
                            if block:
                                handle.write(block)
                                progress.update(len(block))
                if file_sha256(partial) != part["sha256"]:
                    raise ValueError(f"Part {index} is incomplete or changed. Run this cell again.")
                partial.replace(target)
        partial = archive_path.with_suffix(".partial")
        with partial.open("wb") as output:
            for index in range(1, len(parts) + 1):
                with (root / f"asset.part{index}").open("rb") as source:
                    shutil.copyfileobj(source, output, length=1024 * 1024)
        if file_sha256(partial) != archive_sha256:
            raise ValueError("Download is incomplete or changed. Run this cell again.")
        partial.replace(archive_path)
    extract_verified_assets(archive_path, root, archive_sha256, expected_files)
    print("Verified final model and original 9000-post data; ready for inference.")
    return root / "data", root / "seed_42_model"

# Private identifiers may be stored as Colab Secrets instead of notebook text.
try:
    from google.colab import userdata
    for index in range(1, 4):
        key = f"CGSLR_ASSET_PART{index}_FILE_ID"
        if not os.environ.get(key):
            value = userdata.get(key)
            if value:
                os.environ[key] = value
except Exception:
    pass

ASSET_CONFIG = {
    "archive_sha256": "4e5b21501621d933089a3bfedeafd1366296169ba5207d692212c81d0d7fa975",
    "expected_files": {
        "seed_42_model/config.json": "b0d6bfc8cb80089a4df54ddaf13fb1b726ca09f5f28536f793d56afdab708858",
        "seed_42_model/model.safetensors": "0eeb8b16aef6a5df3045c2924d4701305f0ca08dc85d8a66efc3daff33b253d6",
        "seed_42_model/tokenizer.json": "8b79639ec74b46604e730f505186eaafb1006d2fd00f2c4930d168bb7f894680",
        "seed_42_model/tokenizer_config.json": "e1c2a61a99bda00f6c55303a210b30e2f92dcf8b555e215812e2eb583e177ffd",
        "data/evaluation_inputs.csv": "320291db23c55be1e8b725e7a2def90f10c9cda1105dd18633cd21097b20102a",
        "data/evaluation_labels.csv": "a5c104ca71c3120e76f4fbe50e07dd40ac55838e674763dd397b4f886ab50d32",
        "data/phase2_inputs.csv": "0ba48797f3ab70f162c7aa6af7e89debf89b069948fa9385d6b6155bee14bb04",
        "data/selection_manifest.csv": "701a903485887bf2d4c62d7771183debeee9c3e10f1eb335587d1b9a25caa092"
    },
    "parts": [
        {
            "name": "final_seed42_existing9000_assets.zip.part1",
            "sha256": "502b28af8f86c7e19f3277e787d2d0e1bb32eb39cd30571456ad1c1ea9dd9677",
            "size": 94371840,
            "file_id": os.environ.get("CGSLR_ASSET_PART1_FILE_ID")
        },
        {
            "name": "final_seed42_existing9000_assets.zip.part2",
            "sha256": "1da666c3eeec2bfc4d46651b3e97cb6513553d6905101beed7832bf78390ac2b",
            "size": 94371840,
            "file_id": os.environ.get("CGSLR_ASSET_PART2_FILE_ID")
        },
        {
            "name": "final_seed42_existing9000_assets.zip.part3",
            "sha256": "a66335411d8b626d3fa5ad3febf3c5f47a08033e870882679ed15363edcdb97c",
            "size": 68082028,
            "file_id": os.environ.get("CGSLR_ASSET_PART3_FILE_ID")
        }
    ]
}
DATA_SOURCE, MODEL_DIR = prepare_assets(
    DRIVE_ROOT, ASSET_CONFIG["parts"], ASSET_CONFIG["archive_sha256"], ASSET_CONFIG["expected_files"]
)


## 3. 추론 함수
입력 해시·ID·클래스 수를 검증하고, 모델 해시와 실행 설정을 저장합니다. 원문은 행 순서가 아닌 example_id로 결합합니다.

In [ ]:
"""Inference-only evaluation of the independent 9,000 posts with the final model."""

import hashlib
import importlib.metadata
import io
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import softmax
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

LABELS = ["Depression", "Neutral", "Happy"]
TEMPERATURE = 1.4801950079829693
THRESHOLD = 0.70
MAX_LENGTH = 256
BATCH_SIZE = 64
OLD_MODEL_SHA256 = "8b123d526cfd32c6ad7c03d58bc629aa6978df9d750fbcac2f07b9f4702e489b"
FINAL_MODEL_FILES = {
    "config.json": "b0d6bfc8cb80089a4df54ddaf13fb1b726ca09f5f28536f793d56afdab708858",
    "model.safetensors": "0eeb8b16aef6a5df3045c2924d4701305f0ca08dc85d8a66efc3daff33b253d6",
    "tokenizer.json": "8b79639ec74b46604e730f505186eaafb1006d2fd00f2c4930d168bb7f894680",
    "tokenizer_config.json": "e1c2a61a99bda00f6c55303a210b30e2f92dcf8b555e215812e2eb583e177ffd",
}


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def atomic_csv(path, frame):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def lock_manifest(output, manifest):
    output.mkdir(parents=True, exist_ok=True)
    path = output / "experiment_manifest.json"
    if path.exists():
        if json.loads(path.read_text()) != manifest:
            raise ValueError("Input/model/runtime settings changed. Preserve this folder and use a new output directory.")
    else:
        if list(output.glob("*.csv")) or list(output.glob("batches/*.json")):
            raise ValueError("Existing results without a matching manifest; refusing to mix runs.")
        atomic_json(path, manifest)


def read_data(source, expected_hashes):
    """Read only data files, never load or execute the old bundle's model/code."""
    frames = {}
    for name, expected in expected_hashes.items():
        if source.is_file():
            with zipfile.ZipFile(source) as archive:
                candidates = [n for n in archive.namelist() if n.endswith("/data/" + name)]
                if len(candidates) != 1:
                    raise ValueError(f"Archive must contain exactly one data/{name}")
                payload = archive.read(candidates[0])
        else:
            payload = (source / name).read_bytes()
        if hashlib.sha256(payload).hexdigest() != expected:
            raise ValueError(f"9000-post source hash mismatch: {name}")
        frames[name] = pd.read_csv(io.BytesIO(payload), keep_default_na=False, dtype={"example_id": str})
    inputs = frames["evaluation_inputs.csv"]
    for name, frame in frames.items():
        if len(frame) != 9000 or frame.example_id.eq("").any() or frame.example_id.duplicated().any():
            raise ValueError(f"Expected 9000 distinct nonempty IDs: {name}")
        if set(frame.example_id) != set(inputs.example_id):
            raise ValueError(f"ID mismatch: {name}")
    if inputs.text.str.strip().eq("").any():
        raise ValueError("Empty Phase 1 text")
    labels = frames["evaluation_labels.csv"]
    labels["label_str"] = labels.label_str.replace({label + "_Group": label for label in LABELS})
    expected_ids = labels.label_str.map({label: i for i, label in enumerate(LABELS)})
    if expected_ids.isna().any() or not np.array_equal(expected_ids.to_numpy(), labels.label.to_numpy()):
        raise ValueError("Reference label strings and numeric IDs disagree")
    if labels.label_str.value_counts().to_dict() != dict.fromkeys(LABELS, 3000):
        raise ValueError("Expected 3000 posts per class")
    raw = frames["phase2_inputs.csv"].rename(columns={
        "was_truncated": "phase2_input_was_truncated",
        "full_character_count": "phase2_full_character_count",
    })
    if raw.phase2_original_text.str.strip().eq("").any():
        raise ValueError("Empty Phase 2 original text")
    # The original-text CSV is in a different order: joins must always use IDs.
    return inputs[["example_id", "text"]], labels, raw


def resolve_paths(drive_root, source_override=None, model_override=None):
    project = drive_root / "confidence_guided_llm_reasoning"
    model = Path(model_override) if model_override else project / "distilbert_classification_fine_tuned/seed_42_model"
    if not (model / "config.json").is_file():
        raise FileNotFoundError(
            f"Final seed42 model not found: {model}\n"
            "Set MODEL_DIR to the existing final seed42 folder. Do not use the old 9000-package checkpoint. "
            "No training or model download is performed by this notebook."
        )
    if source_override:
        source = Path(source_override)
    else:
        source = drive_root / "CGSLR_New_Holdout_9000_Run_Package.zip"
        if not source.exists():
            candidates = list(drive_root.rglob("CGSLR_New_Holdout_9000_Run_Package.zip"))
            if len(candidates) != 1:
                raise FileNotFoundError(
                    "Could not uniquely locate the additional 9000-post ZIP. Set DATA_SOURCE to its Drive path "
                    "or to its extracted same_source_evaluation/data folder."
                )
            source = candidates[0]
    if not source.exists():
        raise FileNotFoundError(source)
    return source, model


def model_files(model_dir):
    weights = sorted(model_dir.glob("*.safetensors"))
    if not weights:
        weights = sorted(model_dir.glob("pytorch_model*.bin"))
    if not weights:
        raise FileNotFoundError(f"No model weights in {model_dir}")
    metadata = [p for p in model_dir.iterdir() if p.is_file() and
                (p.suffix == ".json" or p.name in {"vocab.txt", "tokenizer.model"})]
    hashes = {p.name: sha256(p) for p in sorted(set(weights + metadata))}
    if OLD_MODEL_SHA256 in hashes.values():
        raise ValueError("This is the old operational checkpoint, not the final seed42 model.")
    if hashes != FINAL_MODEL_FILES:
        raise ValueError("Model/tokenizer files differ from the received final seed42 package.")
    return hashes


def check_label_order(names):
    normalized = [{label + "_Group": label for label in LABELS}.get(name, name) for name in names]
    if normalized != LABELS:
        raise ValueError(f"Unexpected checkpoint label order: {names}")
    return normalized


def prediction_rows(batch, logits):
    logits = np.asarray(logits, dtype=np.float32)
    if logits.shape != (len(batch), 3) or not np.isfinite(logits).all():
        raise ValueError("Invalid classifier logits")
    calibrated = softmax(logits / TEMPERATURE, axis=1)
    raw = softmax(logits, axis=1)
    rows = []
    for identifier, z, p, p_raw in zip(batch.example_id, logits, calibrated, raw):
        idx = int(p.argmax())
        rows.append({
            "example_id": identifier,
            "phase1_label": LABELS[idx],
            "phase1_confidence": float(p[idx]),
            "phase1_raw_confidence": float(p_raw.max()),
            "phase1_routed": bool(p[idx] < THRESHOLD),
            "temperature": TEMPERATURE,
            "routing_threshold": THRESHOLD,
            **{f"phase1_logit_{label.lower()}": float(z[i]) for i, label in enumerate(LABELS)},
            **{f"phase1_probability_{label.lower()}": float(p[i]) for i, label in enumerate(LABELS)},
        })
    return rows


def verify_batch(saved, expected_ids, fingerprint):
    if saved.get("run_fingerprint") != fingerprint:
        raise ValueError("Cached batch belongs to a different run")
    rows = saved["predictions"]
    if [r["example_id"] for r in rows] != list(expected_ids):
        raise ValueError("Cached batch ID/order mismatch")
    for row in rows:
        c = row["phase1_confidence"]
        if (row["phase1_label"] not in LABELS or not np.isfinite(c) or not 0 <= c <= 1
                or not isinstance(row["phase1_routed"], bool)
                or row["phase1_routed"] != (c < THRESHOLD)):
            raise ValueError("Invalid cached prediction")


def export_results(output, inputs, labels, raw, predictions, manifest):
    if predictions.example_id.tolist() != inputs.example_id.tolist():
        raise ValueError("Prediction IDs/order differ from the 9000 inputs")
    final = inputs.merge(predictions, on="example_id", validate="one_to_one")
    final = final.merge(labels[["example_id", "label_str"]].rename(columns={"label_str": "target_label"}),
                        on="example_id", validate="one_to_one")
    final = final.merge(raw, on="example_id", validate="one_to_one")
    routed = final.phase1_routed.astype(bool)
    errors = final.phase1_label.ne(final.target_label)
    summary = {
        "dataset": "existing_same_source_9000_final_pipeline",
        "total_rows": len(final), "routed_rows": int(routed.sum()),
        "phase1_accuracy": float(accuracy_score(final.target_label, final.phase1_label)),
        "phase1_macro_f1": float(f1_score(final.target_label, final.phase1_label, labels=LABELS, average="macro")),
        "errors": int(errors.sum()), "routed_errors": int((routed & errors).sum()),
        "coverage": float((~routed).mean()), "routing_rate": float(routed.mean()),
        "accepted_error_rate": float(errors[~routed].mean()) if (~routed).any() else None,
        "temperature": TEMPERATURE, "routing_threshold": THRESHOLD,
    }
    # The full file preserves the denominator; Phase 2 consumes only routed rows.
    final.loc[~routed, "phase2_original_text"] = ""
    atomic_csv(output / "phase1_predictions.csv", final.drop(columns="phase2_original_text"))
    atomic_csv(output / "final_phase1_reasoning_input.csv", final)
    atomic_csv(output / "routed_phase2_inputs.csv", final.loc[routed].reset_index(drop=True))
    atomic_json(output / "phase1_summary.json", summary)
    atomic_csv(output / "phase1_summary.csv", pd.DataFrame([summary]))
    report = classification_report(final.target_label, final.phase1_label, labels=LABELS, output_dict=True, zero_division=0)
    atomic_json(output / "classification_report.json", report)
    matrix = confusion_matrix(final.target_label, final.phase1_label, labels=LABELS)
    atomic_csv(output / "confusion_matrix.csv", pd.DataFrame(matrix, columns=LABELS).assign(target_label=LABELS))
    atomic_json(output / "phase2_handoff.json", {
        "input_path": str(output / "final_phase1_reasoning_input.csv"),
        "input_sha256": sha256(output / "final_phase1_reasoning_input.csv"),
        "expected_total_rows": len(final), "expected_routed_rows": int(routed.sum()),
        "planned_methods": ["Llama 3 Direct", "Llama 3 CoT", "Llama 3 SELF-DISCOVER v6c"],
        "phase2_executed": False, "phase1_manifest": manifest,
    })
    return summary


def run_phase1(source, model_dir, output, expected_hashes):
    import torch
    import transformers
    from tqdm.auto import tqdm
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime in Colab before running Phase 1.")
    inputs, labels, raw = read_data(source, expected_hashes)
    hashes = model_files(model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True).float().cuda().eval()
    # The source seed42 notebook uses 0=Depression, 1=Neutral, 2=Happy.
    names = [model.config.id2label.get(i) for i in range(3)]
    if model.config.num_labels != 3:
        raise ValueError(f"Unexpected checkpoint label order: {names}")
    check_label_order(names)
    if model.config.model_type != "distilbert":
        raise ValueError("Expected a DistilBERT sequence classifier")
    manifest = {
        "protocol": "final_seed42_existing9000_phase1_v1",
        "source_data_sha256": expected_hashes, "model_file_sha256": hashes,
        "model_dir": str(model_dir), "label_order": LABELS,
        "checkpoint_id2label": names,
        "temperature": TEMPERATURE, "threshold": THRESHOLD,
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "precision": "fp32",
        "training_performed": False, "calibration_fitted": False, "threshold_searched": False,
        "torch": torch.__version__, "transformers": transformers.__version__,
        "numpy": np.__version__, "scipy": importlib.metadata.version("scipy"),
        "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
        "attention_implementation": getattr(model.config, "_attn_implementation", None),
    }
    lock_manifest(output, manifest)
    fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    batches = output / "batches"
    batches.mkdir(exist_ok=True)
    all_rows = []
    completed = sum((batches / f"batch_{s:05d}.json").exists() for s in range(0, len(inputs), BATCH_SIZE))
    print(f"Saved batches: {completed}; total batches: {(len(inputs) + BATCH_SIZE - 1) // BATCH_SIZE}")
    for start in tqdm(range(0, len(inputs), BATCH_SIZE), desc="Final DistilBERT / 9000"):
        batch = inputs.iloc[start:start + BATCH_SIZE]
        path = batches / f"batch_{start:05d}.json"
        if path.exists():
            saved = json.loads(path.read_text())
        else:
            encoded = tokenizer(batch.text.tolist(), padding=True, truncation=True,
                                max_length=MAX_LENGTH, return_tensors="pt", return_token_type_ids=False)
            tokens = int(encoded["attention_mask"].sum())
            encoded = {k: v.cuda() for k, v in encoded.items()}
            torch.cuda.synchronize()
            began = time.perf_counter()
            with torch.inference_mode():
                logits = model(**encoded).logits.float().cpu().numpy()
            torch.cuda.synchronize()
            saved = {"run_fingerprint": fingerprint, "predictions": prediction_rows(batch, logits),
                     "forward_seconds": time.perf_counter() - began, "input_tokens": tokens}
            atomic_json(path, saved)
        verify_batch(saved, batch.example_id, fingerprint)
        all_rows.extend(saved["predictions"])
    summary = export_results(output, inputs, labels, raw, pd.DataFrame(all_rows), manifest)
    del model, tokenizer
    torch.cuda.empty_cache()
    print("Phase 1 complete. Llama generation has NOT started.")
    print("Saved:", output)
    return pd.DataFrame([summary])


## 4. 입력 확인
9,000건, 클래스별 3,000건인지 확인합니다. 이 셀에서는 모델 추론을 실행하지 않습니다.

In [ ]:
EXPECTED_DATA_HASHES = {
    "evaluation_inputs.csv": "320291db23c55be1e8b725e7a2def90f10c9cda1105dd18633cd21097b20102a",
    "evaluation_labels.csv": "a5c104ca71c3120e76f4fbe50e07dd40ac55838e674763dd397b4f886ab50d32",
    "phase2_inputs.csv": "0ba48797f3ab70f162c7aa6af7e89debf89b069948fa9385d6b6155bee14bb04",
    "selection_manifest.csv": "701a903485887bf2d4c62d7771183debeee9c3e10f1eb335587d1b9a25caa092"
}
DATA_PATH, FINAL_MODEL_PATH = resolve_paths(DRIVE_ROOT, DATA_SOURCE, MODEL_DIR)
checked_inputs, checked_labels, checked_raw = read_data(DATA_PATH, EXPECTED_DATA_HASHES)
verified_model_hashes = model_files(FINAL_MODEL_PATH)
checkpoint_config = json.loads((FINAL_MODEL_PATH / "config.json").read_text())
check_label_order([checkpoint_config["id2label"][str(i)] for i in range(3)])
print("Data:", DATA_PATH)
print("Model:", FINAL_MODEL_PATH)
print("Output:", OUTPUT_DIR)
print("Rows:", len(checked_inputs))
display(checked_labels.label_str.value_counts().rename("rows").to_frame())
del checked_inputs, checked_labels, checked_raw


## 5. 최종 모델 분류 실행
중단 후에는 같은 GPU 종류·환경·경로에서 위 셀부터 실행하면 저장된 배치를 건너뜁니다. 설정이 달라지면 결과를 섞지 않고 멈춥니다.

In [ ]:
summary = run_phase1(DATA_PATH, FINAL_MODEL_PATH, OUTPUT_DIR, EXPECTED_DATA_HASHES)
display(summary)


## 6. 다음 단계에 넘길 파일

- `final_phase1_reasoning_input.csv`: 전체 9,000건 예측·정답 및 라우팅된 사례의 원문. 다음 Llama 3 코드의 입력.
- `routed_phase2_inputs.csv`: 새로 라우팅된 사례만 별도로 저장.
- `phase1_summary.csv`: 정확도, Macro F1, 라우팅 건수.
- `experiment_manifest.json`: 모델 해시 및 설정 기록.
- `phase2_handoff.json`: 다음 단계 입력 경로·해시·건수. 정답은 평가에만 사용하며 프롬프트 입력으로 사용하지 않습니다.

분류 결과표를 공유하면 새 라우팅 수를 기준으로 다음 노트북을 연결합니다. 이전 라우팅 137건이나 본 실험 218건을 강제하지 않습니다.


In [ ]:
print("Next-stage input:", OUTPUT_DIR / "final_phase1_reasoning_input.csv")
print("Handoff:", OUTPUT_DIR / "phase2_handoff.json")
print("No Llama model was run in this notebook.")
